# Traductor de txt a SQL

**Índice**   
1. [Imports](#imports)
2. [Cargamos los modelos](#cargamos-los-modelos)
3. [Creamos la base de datos](#creamos-la-base-de-datos)
4. [Mapeo de columnas](#mapeo-de-columnas)
5. [Sinonimos/palabras clave (ES/EN)](#sinonimos--palabras-clave-esen)
6. [Metricas](#metricas)
7. [Agrupaciones temporales](#agrupaciones-temporales)
8. [Funcion que detecta el idioma](#funcion-que-detecta-el-idioma) .
9. [Filtro de fecha](#filtro-de-fecha)
10. [Filtro minus y mayus](#filtro-minus-y-mayus)
11. [Filtro agrupaciones](#filtro-agrupaciones)
12. [Detección de where](#detección-de-where).
13. [Detección de group](#detección-de-group).
14. [Detección de filtro](#detección-de-filtro).
15. [Generador de SQL](#generador-de-SQL)
16. [Pruebas](#pruebas)

## Imports

In [914]:
import re
import spacy
from langdetect import detect
import pandas as pd

## Cargamos los modelos 

In [915]:
# Modelos
nlp_es = spacy.load("es_core_news_sm") # Español
nlp_en = spacy.load("en_core_web_sm") # English

## Creamos la base de datos

In [916]:
# Cargar los CSVs (ajusta las rutas a tus archivos locales)
clientes = pd.read_csv("../data/clientes_ecommerce.csv")
transacciones = pd.read_csv("../data/transacciones_ecommerce.csv")

In [917]:
df = pd.merge(transacciones, clientes, on="id_cliente", how="outer")
TABLE_NAME = "merge_transaccion_cliente"
# IMPORTANTE: en tu df mergeado las columnas son las del CSV, aquí asumo que usas las españolas.

In [918]:
df

,id_transaccion,id_cliente,fecha_compra,producto,categoria_producto,precio_unitario,cantidad,importe_total,metodo_pago,coste_envio,coste_fabricacion,nombre,apellidos,email,pais,ciudad,edad,genero
0,1308.0,1,2024-03-21,Xiaomi 13,Móviles,1269.37,2.0,2538.74,bizum,6.64,723.00,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
1,1423.0,1,2023-09-04,Apple Watch Series 9,Relojes inteligentes,561.03,1.0,561.03,paypal,8.13,301.15,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
2,7682.0,1,2023-08-16,Auriculares Sony WH-1000XM5,Accesorios,224.99,1.0,224.99,tarjeta,5.90,67.77,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
3,8220.0,1,2024-10-26,Lenovo ThinkPad X1 Carbon,Portátiles,1562.94,2.0,3125.88,transferencia,24.97,1166.97,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
4,9001.0,1,2023-02-26,Huawei Watch GT 4,Relojes inteligentes,264.37,3.0,793.11,bizum,7.82,141.60,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15229,386.0,4999,2024-10-05,Apple Watch Series 9,Relojes inteligentes,437.23,2.0,874.46,transferencia,8.11,204.23,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15230,1958.0,4999,2024-09-04,Auriculares Sony WH-1000XM5,Accesorios,10.27,2.0,20.54,paypal,4.76,3.91,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15231,2311.0,4999,2024-02-21,Xiaomi 13,Móviles,739.49,2.0,1478.98,bizum,12.94,464.58,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15232,8452.0,4999,2023-09-20,HP Spectre x360,Portátiles,2215.25,1.0,2215.25,tarjeta,16.57,1340.05,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F


## Mapeo de columnas

In [919]:
COLS = {
    "id_cliente","nombre","apellidos","email","pais","ciudad","edad","genero",
    "id_transaccion","fecha_compra","producto","categoria_producto",
    "precio_unitario","cantidad","importe_total","metodo_pago",
    "coste_envio","coste_fabricacion"
}

## Sinonimos / palabras clave (ES/EN)

In [920]:
# Sinónimos / palabras clave -> columnas (ES/EN)
SYN_TO_COL = {
    "es": {
        # métricas
        "ventas": "importe_total",
        "ingresos": "importe_total",
        "beneficios": "importe_total",
        "facturacion": "importe_total",
        "importe": "importe_total",
        "total": "importe_total",
        "unidades": "cantidad",
        "cantidad": "cantidad",
        "precio": "precio_unitario",
        "envio": "coste_envio",
        "fabricacion": "coste_fabricacion",
        # dimensiones
        "pai": "pais",
        "país": "pais",
        "ciudad": "ciudad",
        "producto": "producto",
        "categoria": "categoria_producto",
        "genero": "genero",
        "edad": "edad",
        "metodo": "metodo_pago",
        "pago": "metodo_pago",
        "fecha": "fecha_compra",
        "compra": "fecha_compra",
        "transaccion": "id_transaccion",
        "pedido": "id_transaccion",
        "cliente": "id_cliente",
    },
    "en": {
        # si el usuario pregunta en inglés, seguimos generando SQL con columnas ES
        # (porque tu dataset está en ES). Solo traducimos la intención.
        "sales": "importe_total",
        "revenue": "importe_total",
        "amount": "importe_total",
        "total": "importe_total",
        "units": "cantidad",
        "quantity": "cantidad",
        "price": "precio_unitario",
        "shipping": "coste_envio",
        "manufacturing": "coste_fabricacion",
        # dimensiones
        "country": "pais",
        "city": "ciudad",
        "product": "producto",
        "category": "categoria_producto",
        "gender": "genero",
        "age": "edad",
        "payment": "metodo_pago",
        "date": "fecha_compra",
        "purchase": "fecha_compra",
        "transaction": "id_transaccion",
        "order": "id_transaccion",
        "client": "id_cliente",
        "customer": "id_cliente",
    }
}

## Meses

In [921]:
MONTHS = {
    "es": {
        "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
        "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
        "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
    },
    "en": {
        "january": 1, "february": 2, "march": 3, "april": 4,
        "may": 5, "june": 6, "july": 7, "august": 8,
        "september": 9, "october": 10, "november": 11, "december": 12,
    }
}

## Mapeo de paises

In [922]:
PAIS_MAP = {
    "mexico": "México",
    "argentina": "Argentina",
    "espana": "España",
    "en reino unido": "Reino Unido",
    "francia": "Francia",
    "portugal": "Portugal",
    "alemania": "Alemania",
    "italia": "Italia",

    # Inglés
    "mexico": "México",
    "argentina": "Argentina",
    "spain": "España",
    "united kingdom": "Reino Unido",
    "uk": "Reino Unido",
    "england": "Reino Unido",
    "france": "Francia",
    "portugal": "Portugal",
    "germany": "Alemania",
    "italy": "Italia"

}

## Mapeo de categoria

In [923]:
CATEGORIA_MAP = {
    "accesorios": "Accesorios",
    "accesorio": "Accesorios",

    "reloj": "Relojes inteligentes",
    "relojes": "Relojes inteligentes",
    "reloj inteligente": "Relojes inteligentes",
    "smartwatch": "Relojes inteligentes",

    "movil": "Móviles",
    "moviles": "Móviles",
    "telefono": "Móviles",
    "telefonos": "Móviles",
    "smartphone": "Móviles",

    "portatil": "Portátiles",
    "portatiles": "Portátiles",
    "laptop": "Portátiles",
    "ordenador": "Portátiles",
    "computadora": "Portátiles",
}

## Metricas

FALTAN ¿Varianza?

In [924]:
AGG_WORDS = {
    "es": {
        "avg": {"promedio", "media", "promediar"},
        "sum": {"suma", "total", "sumar", "sumatorio"},
        "count": {"cuantos", "cuantas", "numero", "numeros", "conteo", "contar"},
        "max": {"maximo", "maxima", "mayor", "pico", "tope"},
        "min": {"minimo", "minima", "menor", "bajo"},
        "median": {"mediana", "percentil", "percentil 50"},
        "mode": {"moda", "mas frecuente", "frecuente"},
        "std": {"desviacion", "desviacion estandar", "variacion"},
    },
    "en": {
        "avg": {"average", "avg", "mean"},
        "sum": {"sum", "total"},
        "count": {"count", "how", "many", "number"},
        "max": {"max", "maximum", "highest", "top"},
        "min": {"min", "minimum", "lowest"},
        "median": {"median", "percentile"},
        "mode": {"mode", "most frequent"},
        "std": {"std", "stddev", "standard deviation"}
    }
}

## Agrupaciones temporales

In [925]:
TIME_GROUP_WORDS = {
    "es": {
        "quarter": {"trimestre", "trimestral", "trimestralmente"},
        "month": {"mes", "mensual"},
        "year": {"año", "ano", "anual"},
    },
    "en": {
        "quarter": {"quarter", "qtr"},
        "month": {"month", "monthly"},
        "year": {"year", "yearly", "annual"},
    }
}

## Mapeo para rankings

In [1013]:
RANKING_WORDS = {
    "es": {
        "cantidad": {
            "mas vendido",
            "más vendido",
            "mas vendidos",
            "más vendidos",
            "top vendidos",
            "productos mas vendidos",
            "mayor volumen",
            "mayor volumen de ventas",
        },
        "importe_total": {
            "mayores ventas",
            "mas ingresos",
            "más ingresos",
            "mayor facturacion",
            "ingresos mas altos",
        },
    },
    "en": {
        "cantidad": {
            "best selling",
            "most sold",
            "top selling",
            "highest volume",
            "highest sales volume",
        },
        "importe_total": {
            "highest revenue",
            "top revenue",
            "most revenue",
            "top sales",
        },
    },
}

## Funcion que detecta el idioma

In [927]:
def detectar_idioma(texto: str):
    lang = detect(texto)
    if lang == "es":
        return nlp_es(texto), "es"
    elif lang == "en":
        return nlp_en(texto), "en"
    else:
        raise ValueError(f"Idioma no soportado: {lang}")

## Prepocesamiento del texto

### Quitar tíldes

In [928]:
import unicodedata

def strip_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

### Prepocesado minus y mayus

In [929]:
def _normalize_tokens(doc):
    # lemmas en minúscula, sin puntuación/espacios
        return [
        strip_accents(t.lemma_.lower())
        for t in doc
        if not t.is_punct and not t.is_space
    ]

## Filtro de fecha

### Filtro año

In [930]:
def _find_years(texto: str):
    return sorted(set(re.findall(r"\b(2023|2024)\b", texto)))

### SQL año

In [931]:
def _year_range_condition_pg(years):
    # years es lista de strings [“2023”] o [“2023",“2024"]
    start_y = min(years)
    end_y = max(years)
    return (
        f"fecha_compra BETWEEN '{start_y}-01-01' AND '{end_y}-12-31'"
    )

### Rango meses

In [932]:
def _find_month_ranges(texto: str, idioma: str):
    texto = texto.lower()
    pattern = (
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)"
        r"\s+(a|y|hasta|to|and)\s+"
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)"
        r"(?:\s+(\d{4}))?"
    )
    matches = re.findall(pattern, texto)
    # Devuelve mes_inicio, mes_fin, año (como int o None)
    result = []
    for m_start, _, m_end, year in matches:
        y = int(year) if year else None
        result.append((m_start, m_end, y))
    return result

### SQL rango meses

In [933]:
from calendar import monthrange

def _month_range_condition_pg(start_month, end_month, year):
    start_date = f"{year}-{start_month:02d}-01"
    # último día del mes final
    last_day = monthrange(int(year), end_month)[1]
    end_date = f"{year}-{end_month:02d}-{last_day}"
    return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"


### Filtro un mes en concreto de un año en concreto

In [934]:
def _find_single_month(texto: str, idioma: str):
    texto = texto.lower()
    for m, num in MONTHS[idioma].items():
        m_match = re.search(rf"\b{m}\b(?:\s+(?:de|del|of))?\s*(\d{{4}})?", texto)
        if m_match:
            year = m_match.group(1)
            return num, year
    return None

### SQL un mes en concreto de un año en concreto

In [935]:
def _single_month_condition_pg(month, year):
    start_date = f"{year}-{month:02d}-01"
    end_day = monthrange(int(year), month)[1]
    end_date = f"{year}-{month:02d}-{end_day}"
    return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"

### Fechas relativas

In [956]:
from calendar import monthrange
import re

#def _last_day_of_month(year, month):
#    if month == 12:
#        return 31
#    return (datetime(year, month + 1, 1) - timedelta(days=1)).day

def _last_day_of_month(year, month):
    return monthrange(year, month)[1]

def _relative_time_condition(doc, idioma: str, available_years=[2023, 2024]):
    texto = strip_accents(doc.text.lower())

    years_in_text = _find_years(texto)
    year = int(years_in_text[0]) if years_in_text else None

    # TRIMESTRE
    m_quarter = re.search(r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+(trimestre|quarter)",texto)
    if m_quarter:
        tipo_raw = m_quarter.group(1)
        tipo = "last" if "ultim" in tipo_raw or tipo_raw == "last" else "first"
        if not year:
            year = max(available_years) if tipo == "last" else min(available_years)
        start_month, end_month = (1, 3) if tipo == "first" else (10, 12)
        start_date = f"{year}-{start_month:02d}-01"
        end_day = _last_day_of_month(year, end_month)
        end_date = f"{year}-{end_month:02d}-{end_day}"
        return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"

    # Meses relativos
    m_months = re.search(r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+(meses?|months?)", texto)
    if m_months:
        tipo_raw = m_months.group(1)
        n_months = int(m_months.group(2)) if m_months.group(2) else 1
        n_months = min(max(n_months, 1), 12)
        tipo = "last" if "ultim" in tipo_raw or tipo_raw == "last" else "first"
        if not year:
            year = max(available_years) if tipo == "last" else min(available_years)
        if tipo == "first":
            start_month, end_month = 1, n_months
        else:
            start_month, end_month = 12 - n_months + 1, 12
        start_date = f"{year}-{start_month:02d}-01"
        end_day = _last_day_of_month(year, end_month)
        end_date = f"{year}-{end_month:02d}-{end_day}"
        return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"

    # Año completo
    m_year = re.search(r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+(ano|año|year)", texto)
    if m_year:
        tipo_raw = m_year.group(1)
        tipo = "last" if "ultim" in tipo_raw or tipo_raw == "last" else "first"
        if not year:
            year = max(available_years) if tipo == "last" else min(available_years)
        return f"fecha_compra BETWEEN '{year}-01-01' AND '{year}-12-31'"

    return None

## Ranking

### Filtro ranking

In [937]:
def detectar_ranking(tokens, idioma):
    top_words = {
        "es": {"top", "mejores", "mayores", "ranking"},
        "en": {"top", "best", "highest", "ranking"},
    }

    if any(t in tokens for t in top_words[idioma]):
        return True
    return False

### Ranking

In [938]:
def detectar_ranking_semantico(texto: str, idioma: str):
    t = strip_accents(texto.lower())

    for metric, phrases in RANKING_WORDS[idioma].items():
        for p in phrases:
            if p in t:
                return "sum", metric

    return None

### Detectar N top

In [939]:
def detectar_limit(texto: str):
    m = re.search(r"\btop\s+(\d+)", texto.lower())
    if m:
        return int(m.group(1))
    return 5  # default razonable

## Dimesión del ranking

In [1005]:
def detectar_dimension_ranking(tokens, idioma):
    tokens_norm = [strip_accents(t.lower()) for t in tokens]
    syn_norm = {strip_accents(k.lower()): v for k, v in SYN_TO_COL[idioma].items()}

    for tok in tokens_norm:
        if tok in syn_norm:
            col = syn_norm[tok]
            if col in {"pais", "producto", "id_cliente", "ciudad", "categoria_producto"}:
                return col
    return None

## Filtro agrupaciones

In [940]:
def detectar_agregacion(tokens, idioma):
    # default: None (si no pide nada, se puede devolver *)
    for agg, words in AGG_WORDS[idioma].items():
        if any(w in tokens for w in words):
            return agg
    return None

## Detección de where

In [941]:
def detectar_metricas(tokens, idioma):
    # Busca la primera métrica "razonable"
    # Si menciona ventas/importe -> importe_total; unidades -> cantidad; etc.
    for tok in tokens:
        if tok in SYN_TO_COL[idioma]:
            col = SYN_TO_COL[idioma][tok]
            if col in {"importe_total", "cantidad", "precio_unitario", "coste_envio", "coste_fabricacion"}:
                return col
    # fallback: si menciona ventas/total en cualquier parte del texto, asumimos importe_total
    texto = " ".join(tokens)
    if re.search(r"\b(venta|ventas|revenue|ingresos|total|totales)\b", texto.lower()):
        return "importe_total"
    return None

## Detección de group

In [942]:
def detectar_groupbys(tokens, idioma, filtros=None):
    group_cols = []
    skip_time_group = False
    auto_time_group = None
    
    if filtros:
        for f in filtros:
            m = re.search(r"BETWEEN '(\d{4})-(\d{2})-(\d{2})' AND '(\d{4})-(\d{2})-(\d{2})'", f)
            if m:
                start_year, start_month, start_day, end_year, end_month, end_day = map(int, m.groups())
                if (end_year == start_year) and ((end_month - start_month + 1) < 12):
                    skip_time_group = True
                    # decidir granularity según meses
                    months_span = end_month - start_month + 1
                    if months_span == 1:
                        auto_time_group = "month"
                    elif months_span == 3:
                        auto_time_group = "quarter"
                    else:
                        auto_time_group = "year"
                    break

    # Tiempo
    time_map = {
        "quarter": ("date_trunc('quarter', fecha_compra)", "trimestre"),
        "month": ("date_trunc('month', fecha_compra)", "mes"),
        "year": ("date_trunc('year', fecha_compra)", "anio"),
    }

    tokens_norm = [strip_accents(t.lower()) for t in tokens]

    # Agrupaciones temporales
    if skip_time_group and auto_time_group:
        expr, _ = time_map[auto_time_group]
        group_cols.append(expr)
    else:
        for granularity, (expr, alias) in time_map.items():
            if any(t in [strip_accents(w) for w in TIME_GROUP_WORDS[idioma][granularity]] for t in tokens_norm):
                group_cols.append(expr)
                break

    # Normalizar SYN_TO_COL
    syn_norm = {strip_accents(k.lower()): v for k, v in SYN_TO_COL[idioma].items()}

    # Detectar dimensiones mencionadas usando trigger ("por"/"by")
    trigger_words = {"es": {"por", "el"}, "en": "by"}
    trigger = trigger_words[idioma]

    for i, tok in enumerate(tokens_norm):
        if tok == trigger and i + 1 < len(tokens_norm):
            next_tok = tokens_norm[i + 1]
            if next_tok in syn_norm:
                col = syn_norm[next_tok]
                if col in COLS and col not in group_cols:
                    group_cols.append(col)

    return group_cols

## Detección de filtro

In [943]:
def detectar_filtros(doc, tokens, idioma):
    has_relative = False
    has_month_range = False
    has_single_month = False
    
    where = []
    text = strip_accents(doc.text.lower())  # todo en minúscula y sin tildes

    # Fechas relativas (prioridad absoluta)
    relative = _relative_time_condition(doc, idioma)
    if relative:
        where.append(relative)
        has_relative = True

    # Rango de meses
    month_ranges = _find_month_ranges(text, idioma)
    if month_ranges:
        for m_start, m_end, year in month_ranges:
            if not year:
                years_in_text = _find_years(text)
                year = int(years_in_text[0]) if years_in_text else 2024
            start_num = MONTHS[idioma][strip_accents(m_start.lower())]
            end_num = MONTHS[idioma][strip_accents(m_end.lower())]
            where.append(_month_range_condition_pg(start_num, end_num, year))
        has_month_range = True
    
    # Mes en concreto
    single_month = _find_single_month(text, idioma)
    if single_month and not has_month_range:
        month, year = single_month
        if not year:
            years_in_text = _find_years(text)
            year = int(years_in_text[0]) if years_in_text else 2024
        where.append(_single_month_condition_pg(month, year))
        has_single_month = True

    # Año(s) solo si no hay fecha relativa
    if not has_relative and not has_month_range and not has_single_month:
        years = _find_years(text)
        if years:
            where.append(_year_range_condition_pg(years))

    # País / ciudad
    for ent in doc.ents:
        ent_text_norm = strip_accents(ent.text.lower())
        if ent.label_ in {"LOC", "GPE"}:
            span_start = max(ent.start - 2, 0)
            span_end = min(ent.end + 2, len(doc))
            window = " ".join([strip_accents(t.lemma_.lower()) for t in doc[span_start:span_end]])
            if ("ciudad" in window) or ("city" in window):
                where.append(f"ciudad = '{ent_text_norm}'")
            else:
                pais_real = PAIS_MAP.get(ent_text_norm, ent.text)
                where.append(f"pais = '{pais_real}'")

    # Producto / categoría
    m_prod = re.search(r"(producto)\s+([a-z0-9_\-áéíóúñ ]{2,})", text)
    if m_prod:
        val = m_prod.group(2).strip()
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        val = strip_accents(val.lower())

        cat_real = CATEGORIA_MAP.get(val)

        if cat_real:
            where.append(f"categoria_producto = '{cat_real}'")
        else:
            where.append(
                f"categoria_producto LIKE '%{val.replace('\'','\'\'')}%'"
            )

    m_cat = re.search(r"(categor[ií]a)\s+([a-z0-9_\-áéíóúñ ]{2,})", text)
    if m_cat:
        val = m_cat.group(2).strip()
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        val = strip_accents(val.lower())

        cat_real = CATEGORIA_MAP.get(val)

        if cat_real:
            where.append(f"categoria_producto = '{cat_real}'")
        else:
            where.append(
                f"categoria_producto LIKE '%{val.replace('\'','\'\'')}%'"
            )

    # Género
    if re.search(r"\b(masculino|macho?s|varon?es|hombre|hombres|male|m)\b", text):
        where.append("genero IN ('M')")
    if re.search(r"\b(femenino|hembra?s|mujer|mujeres|female|f)\b", text):
        where.append("genero IN ('F')")

    # Edad
    m_gt = re.search(r"(mayores de|mas de|over|older than)\s+(\d{1,3})", text)
    if m_gt:
        where.append(f"edad > {int(m_gt.group(2))}")
    m_lt = re.search(r"(menores de|menos de|under|younger than)\s+(\d{1,3})", text)
    if m_lt:
        where.append(f"edad < {int(m_lt.group(2))}")

    # Dedup
    where_out = []
    seen = set()
    for w in where:
        if w not in seen:
            where_out.append(w)
            seen.add(w)

    return where_out


## Generador de SQL

In [1004]:
def generar_sql(texto: str):
    doc, idioma = detectar_idioma(texto)
    tokens_norm = _normalize_tokens(doc)
        
    # Detectar GROUP BY, agregación y métricas
    where = detectar_filtros(doc, tokens_norm, idioma)

    group_by = detectar_groupbys(tokens_norm, idioma, filtros=where)
    
    agg = detectar_agregacion(tokens_norm, idioma)
    metric = detectar_metricas(tokens_norm, idioma)

    has_ranking = detectar_ranking(tokens_norm, idioma)
    ranking_impl = detectar_ranking_semantico(texto, idioma)
    ranking_dim = detectar_dimension_ranking(tokens_norm, idioma)


    if ranking_impl:
        agg, metric = ranking_impl
    else:
        agg = detectar_agregacion(tokens_norm, idioma)
        metric = detectar_metricas(tokens_norm, idioma)

    if has_ranking and ranking_dim:
        group_by = [ranking_dim]

    if has_ranking and not group_by:
        if "producto" in tokens_norm or "productos" in tokens_norm:
            group_by = ["producto"]
        elif "cliente" in tokens_norm or "clientes" in tokens_norm:
            group_by = ["id_cliente"]
        elif "pais" in tokens_norm or "paises" in tokens_norm:
            group_by = ["pais"]

    # Ajustes razonables por defecto
    if agg == "count" and not metric:
        metric = "id_transaccion"
        
    # Defaults "razonables"
    if agg in {"avg", "sum", "max", "min"} and metric is None:
        metric = "importe_total"
    
    # Si hay GROUP BY pero no se detecta métrica, asumimos SUM(importe_total)
    if not metric and group_by:
        metric = "importe_total"
        agg = "sum"

    if agg is None and metric is not None:
        if metric in {"importe_total", "cantidad"}:
            agg = "sum"

    # FILTROS: directamente los construye detectar_filtros

    alias_metric = None
    # SELECT
    select_parts = []
    if group_by:
        select_parts.extend(group_by)
    
    if agg == "avg":
        alias_metric = f"promedio_{metric}"
        select_parts.append(f"AVG({metric}) AS {alias_metric}")

    elif agg == "sum":
        alias_metric = f"total_{metric}"
        select_parts.append(f"SUM({metric}) AS {alias_metric}")

    elif agg == "max":
        alias_metric = f"max_{metric}"
        select_parts.append(f"MAX({metric}) AS {alias_metric}")

    elif agg == "min":
        alias_metric = f"min_{metric}"
        select_parts.append(f"MIN({metric}) AS {alias_metric}")
    
    elif agg == "median":
        alias_metric = f"mediana_{metric}"
        select_parts.append(f"PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {metric}) AS {alias_metric}")
    
    elif agg == "mode":
        alias_metric = f"moda_{metric}"
        select_parts.append(f"MODE() WITHIN GROUP (ORDER BY {metric}) AS {alias_metric}")
    
    elif agg == "std":
        alias_metric = f"std_{metric}"
        select_parts.append(f"STDDEV_POP({metric}) AS {alias_metric}")

    elif agg == "count":
        # Si pide conteo, contamos transacciones por defecto
        alias_metric = "conteo_transacciones"
        select_parts.append(f"COUNT(DISTINCT id_transaccion) AS {alias_metric}")

    else:
        if not group_by:
        # Sin intención: devuelve columnas principales (evita SELECT *)
            select_parts.append("id_transaccion")
            select_parts.append("fecha_compra")
            select_parts.append("importe_total")
            select_parts.append("cantidad")

    sql = "SELECT " + ", ".join(select_parts) + f" FROM {TABLE_NAME}"

    if where:
        sql += " WHERE " + " AND ".join(where)

    if group_by:
        sql += " GROUP BY " + ", ".join(group_by)

    if has_ranking and alias_metric:
        sql += f" ORDER BY {alias_metric} DESC"
        sql += f" LIMIT {detectar_limit(texto)}"
        
    sql += ";"
    return sql

## Pruebas

LEFT JOIN --> De transacciones a cliente
PROS:
- clientes.id_cliente es PK
- transaccion.id_transacciones es PK
- transaccion.id_cliente es FK
- No hay duplicados reales con LEFT JOIN.

FULL OTER --> Genera filas fantasmas
CONTRA:
- COUNT(id_transaccion)
- SUM(importe_total)
- rankings
- métricas BI

### 1️⃣ Fechas relativas (mes / trimestre / año) ✅

In [957]:
print(generar_sql("¿Cuántas transacciones hubo en el primer trimestre de 2024?"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-03-31' GROUP BY date_trunc('quarter', fecha_compra);


In [946]:
print(generar_sql("Promedio de ventas entre febrero y abril de 2024"))

SELECT date_trunc('quarter', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-02-01' AND '2024-04-30' GROUP BY date_trunc('quarter', fecha_compra);


In [947]:
print(generar_sql("Promedio de ventas de los primeros 2 meses de 2024"))

SELECT date_trunc('month', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


In [731]:
print(generar_sql("Total de ventas de los últimos 6 meses de 2023"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-07-01' AND '2023-12-31' GROUP BY date_trunc('month', fecha_compra);


In [732]:
print(generar_sql("Número de transacciones en el último año"))

SELECT date_trunc('year', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('year', fecha_compra);


In [733]:
print(generar_sql("Ventas del 2023"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [734]:
print(generar_sql("Las ventas en el 2024"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


### 2️⃣ Fechas absolutas + group by temporal ✅

In [735]:
print(generar_sql("Ventas totales en enero de 2024"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-01-31';


In [736]:
print(generar_sql("Número de transacciones en marzo de 2023"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-03-01' AND '2023-03-31';


In [759]:
print(generar_sql("Promedio de ventas entre febrero y abril de 2024"))

SELECT AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-02-01' AND '2024-04-30';


In [760]:
print(generar_sql("Ventas totales entre junio y septiembre de 2023 por mes"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-06-01' AND '2023-09-30' GROUP BY date_trunc('month', fecha_compra);


In [304]:
print(generar_sql("Número de transacciones en 2024 por trimestre"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('quarter', fecha_compra);


In [305]:
print(generar_sql("Promedio de ventas en 2023 por mes"))

SELECT date_trunc('month', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY date_trunc('month', fecha_compra);


### 3️⃣ País / ciudad (mapeo + NER) ✅

In [306]:
print(generar_sql("Cuántas transacciones hubo en España en 2024"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'España';


In [307]:
print(generar_sql("Ventas totales en México por mes en 2023"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'México' GROUP BY date_trunc('month', fecha_compra);


In [951]:
print(generar_sql("Promedio de ventas en Francia en el primero trimestre de 2024"))

SELECT date_trunc('quarter', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-03-31' AND pais = 'Francia' GROUP BY date_trunc('quarter', fecha_compra);


In [309]:
print(generar_sql("Ventas totales en Argentina en los últimos 3 meses de 2023"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-10-01' AND '2023-12-31' AND pais = 'Argentina' GROUP BY date_trunc('month', fecha_compra);


In [958]:
print(generar_sql("Número de transacciones en Portugal por trimestre"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE pais = 'Portugal' GROUP BY date_trunc('quarter', fecha_compra);


In [311]:
print(generar_sql("Ventas por país en 2024"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY pais;


### 4️⃣ Métricas distintas (sum, avg, max, min, std, median) 🟡

In [314]:
print(generar_sql("Promedio de ventas en 2024"))

SELECT AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


In [316]:
print(generar_sql("Ventas máximas en España en 2023"))

SELECT MAX(importe_total) AS max_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España';


In [319]:
print(generar_sql("Mínimo ventas en México en 2024"))

SELECT MIN(importe_total) AS min_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'México';


In [959]:
print(generar_sql("Desviación estándar de ventas en 2023"))

SELECT STDDEV_POP(importe_total) AS std_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [960]:
print(generar_sql("Mediana de ventas en el último trimestre de 2024"))

SELECT date_trunc('quarter', fecha_compra), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY importe_total) AS mediana_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31' GROUP BY date_trunc('quarter', fecha_compra);


In [961]:
print(generar_sql("Moda del ventas en 2023"))

SELECT MODE() WITHIN GROUP (ORDER BY importe_total) AS moda_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


#### NO DEVUELVE EL PRODUCTO MÁS VENDIDO (MAPEAR QUE SI NO DEFINE LA CATEGORIA O PRODUCTO, MUESTRE TODOS)

In [1014]:
print(generar_sql("El producto más vendido en 2023"))

SELECT SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND categoria_producto LIKE '%mas vendido%';


In [1015]:
print(generar_sql("Productos más vendidos del último trimestre de 2023"))

SELECT date_trunc('quarter', fecha_compra), SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-10-01' AND '2023-12-31' GROUP BY date_trunc('quarter', fecha_compra);


### 5️⃣ Count (con y sin group by) ✅

In [325]:
print(generar_sql("Cuántas transacciones hubo en 2023"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [326]:
print(generar_sql("Número de transacciones por mes en 2024"))

SELECT date_trunc('month', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


In [327]:
print(generar_sql("Cuántas transacciones hubo en España por trimestre en 2023"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España' GROUP BY date_trunc('quarter', fecha_compra);


In [328]:
print(generar_sql("Número de pedidos en México en el primer trimestre de 2024"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'México' GROUP BY date_trunc('quarter', fecha_compra);


In [329]:
print(generar_sql("Cuántas transacciones por país en 2024"))

SELECT pais, COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY pais;


### 6️⃣ Ranking / Top N ✅

In [980]:
print(generar_sql("Top 5 productos más vendidos en 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [981]:
print(generar_sql("Top 3 países con mayores ventas en 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY pais ORDER BY total_importe_total DESC LIMIT 3;


In [982]:
print(generar_sql("Top 10 productos por ingresos en 2024"))

SELECT producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_importe_total DESC LIMIT 10;


In [984]:
print(generar_sql("Productos más vendidos del último trimestre de 2023"))

SELECT date_trunc('quarter', fecha_compra), SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-10-01' AND '2023-12-31' GROUP BY date_trunc('quarter', fecha_compra);


In [1010]:
print(generar_sql("Top 5 categorías con mayor facturación en 2024"))

SELECT categoria_producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY categoria_producto ORDER BY total_importe_total DESC LIMIT 5;


In [986]:
print(generar_sql("Top 3 países por volumen de ventas en 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY pais ORDER BY total_importe_total DESC LIMIT 3;


### 7️⃣ Ranking + filtros temporales 🟡

In [987]:
print(generar_sql("Top 5 productos más vendidos en España en 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'España' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [1006]:
print(generar_sql("Top 3 países con más ingresos en el primer trimestre de 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-03-31' GROUP BY pais ORDER BY total_importe_total DESC LIMIT 3;


In [1007]:
print(generar_sql("Top 10 productos por ventas en los últimos 3 meses de 2024"))

SELECT producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_importe_total DESC LIMIT 10;


#### NO HACE TOP // COMPLEJA

In [1008]:
print(generar_sql("Productos más vendidos en México en 2023"))

SELECT SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'México';


In [1009]:
print(generar_sql("Top 5 ciudades con más ventas en 2024"))

SELECT ciudad, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY ciudad ORDER BY total_importe_total DESC LIMIT 5;


### 8️⃣ Filtros demográficos ✅

In [341]:
print(generar_sql("Número de clientes menores de 25 en 2024"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND edad < 25;


In [343]:
print(generar_sql("Ventas totales a mujeres en 2023"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND genero IN ('F');


In [344]:
print(generar_sql("Promedio de ventas de hombres en España en 2024"))

SELECT AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND genero IN ('M');


In [345]:
print(generar_sql("Número de transacciones de clientes mayores de 40 en México"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE pais = 'México' AND edad > 40;


In [367]:
## TOP
print(generar_sql("Ventas totales por género en 2024"))

SELECT genero, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY genero;


### 9️⃣ Producto / categoría ✅

In [361]:
print(generar_sql("Ventas totales del producto movil en 2024"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND categoria_producto = 'Móviles';


In [364]:
print(generar_sql("Número de transacciones de la categoría reloj en 2023"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND categoria_producto = 'Relojes inteligentes';


In [366]:
## TOP
print(generar_sql("Promedio de ventas por categoría en 2024"))

SELECT categoria_producto, AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND categoria_producto LIKE '%%' GROUP BY categoria_producto;


In [1016]:
print(generar_sql("Top 5 productos más vendidos de la categoría accesorios"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE categoria_producto = 'Accesorios' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [370]:
print(generar_sql("Ventas totales por categoría en España en 2023"))

SELECT categoria_producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España' AND categoria_producto LIKE '%%' GROUP BY categoria_producto;


### 🔟 Inglés (para validar bilingüe completo) ✅

In [371]:
print(generar_sql("How many transactions were there in 2024?"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


In [372]:
print(generar_sql("Total sales by country in 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY pais;


In [762]:
print(generar_sql("Average sales per month in 2024"))

SELECT date_trunc('month', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


In [1017]:
print(generar_sql("Top 5 best selling products in 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [972]:
print(generar_sql("Number of transactions in Spain in the first quarter of 2023"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-03-31' AND pais = 'España' GROUP BY date_trunc('quarter', fecha_compra);


In [381]:
print(generar_sql("Total revenue in the last 3 months of 2024"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


In [384]:
generar_sql("How many transactions in the UK in 2023?")

"SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'Reino Unido';"

### 1️⃣1️⃣ Edge cases interesantes 🟡

#### CASCA IDIOMA

In [ ]:
print(generar_sql("Ventas"))

In [386]:
print(generar_sql("Cuántas ventas hubo"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente;


In [387]:
print(generar_sql("Dame las ventas por mes"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente GROUP BY date_trunc('month', fecha_compra);


In [973]:
## IGUAL A LA SEGUNDA, YA QUE SI NO HACE UNA TRANSACCIÓN NO LO CONSIDERAMOS COMO CLIENTE
print(generar_sql("Cuántos clientes tuve")) 

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente;


In [390]:
print(generar_sql("Ventas por país"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente GROUP BY pais;


In [1018]:
print(generar_sql("Ventas en España"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE pais = 'España';


#### CASCA IDIOMA

In [ ]:
print(generar_sql("Top ventas"))

---

---

---

## No debería aparecer la fecha

In [1020]:
print(generar_sql("¿Cuántas transacciones hubo en España en el año 2023?"))

SELECT date_trunc('year', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España' GROUP BY date_trunc('year', fecha_compra);


## Hacer count y mostrar column género

In [1021]:
print(generar_sql("cuantas clientes mujeres hubieron en el 2023"))

SELECT id_transaccion, fecha_compra, importe_total, cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND genero IN ('F');
